# Semantic Model Similarity - Results

Interactive app for the semantic model similarity analysis. It reads the scored results from the lakehouse (written by the **002_semantic_model_similarity** notebook) and renders them below. Run the notebook top to bottom and explore - there is no code to edit.

In [ ]:
def build_report_dependency_payload(models, reports, scans, load_status="available"):
    from collections import Counter
    from urllib.parse import quote

    result = {
        "status": "not_collected",
        "scannedAt": "",
        "scope": "",
        "workspaceCount": 0,
        "incompleteWorkspaces": 0,
        "reportCount": 0,
        "linkedReportCount": 0,
        "byModel": {model_id: [] for model_id in models},
        "otherReports": [],
        "scans": [],
    }
    if load_status != "available":
        result["status"] = load_status
        return result
    if not scans:
        return result
    scan_ids = {row.get("scan_id") for row in scans}
    scan_workspaces = {row.get("report_workspace_id") for row in scans}
    scopes = {row.get("scan_scope") for row in scans}
    if (len(scan_ids) != 1 or not next(iter(scan_ids)) or len(scopes) != 1
            or len(scan_workspaces) != len(scans)
            or any(row.get("scan_id") not in scan_ids for row in reports)):
        result["status"] = "inconsistent"
        return result
    unique_reports = {}
    for row in reports:
        key = (row.get("report_workspace_id"), row.get("report_id"))
        if not all(key) or key[0] not in scan_workspaces:
            result["status"] = "inconsistent"
            return result
        if key in unique_reports and unique_reports[key].get("model_id") != row.get("model_id"):
            result["status"] = "inconsistent"
            return result
        unique_reports[key] = row
    workspace_counts = Counter(key[0] for key in unique_reports)
    if any(row.get("report_count") != workspace_counts[row.get("report_workspace_id")] for row in scans):
        result["status"] = "inconsistent"
        return result
    result["incompleteWorkspaces"] = sum(row.get("scan_status") != "complete" for row in scans)
    result["status"] = "partial" if result["incompleteWorkspaces"] else "complete"
    result["scannedAt"] = str(scans[0].get("scanned_at") or "")
    result["scope"] = str(scans[0].get("scan_scope") or "Unknown scope")
    result["workspaceCount"] = sum(bool(row.get("report_workspace_id")) for row in scans)
    result["scans"] = [{
        "workspace": str(row.get("report_workspace_name") or "Workspace discovery"),
        "status": str(row.get("scan_status") or "unknown"),
        "reportCount": int(row.get("report_count") or 0),
        "errorType": str(row.get("error_type") or ""),
    } for row in scans]
    model_index = {str(model_id).casefold(): model_id for model_id in models}
    for row in unique_reports.values():
        model_id = model_index.get(str(row.get("model_id") or "").casefold())
        workspace_id = str(row["report_workspace_id"])
        report_id = str(row["report_id"])
        report = {
            "id": report_id,
            "name": str(row.get("report_name") or report_id),
            "workspaceId": workspace_id,
            "workspace": str(row.get("report_workspace_name") or workspace_id),
            "url": "https://app.powerbi.com/groups/" + quote(workspace_id, safe="") + "/reports/" + quote(report_id, safe=""),
            "modelId": model_id or "",
            "crossWorkspace": False,
        }
        if model_id is not None and row.get("report_type") == "PowerBIReport":
            report["crossWorkspace"] = workspace_id.casefold() != str(models[model_id].get("workspace_id") or "").casefold()
            result["byModel"][model_id].append(report)
            result["linkedReportCount"] += 1
        else:
            report["reason"] = {
                "missing_dataset_id": "Missing model binding",
                "unsupported_report_type": "Unsupported report type",
            }.get(row.get("binding_status"), "Model outside catalog")
            result["otherReports"].append(report)
    for group in list(result["byModel"].values()) + [result["otherReports"]]:
        group.sort(key=lambda report: (report["name"].casefold(), report["workspace"].casefold(), report["id"]))
    result["reportCount"] = len(unique_reports)
    return result


def render_results():
    """Load the latest results from the lakehouse and render the interactive app."""
    import hashlib
    import json
    import re
    from datetime import datetime, timezone

    import pandas as pd
    from pyspark.sql.utils import AnalysisException

    def load_delta(table_name):
        return spark.table(table_name).toPandas()

    models_df = load_delta("semantic_models")
    tables_df = load_delta("semantic_model_tables")
    columns_df = load_delta("semantic_model_columns")
    relationships_df = load_delta("semantic_model_relationships")
    measures_df = load_delta("semantic_model_measures")
    datasources_df = load_delta("semantic_model_datasources")
    pairs_df = load_delta("semantic_model_similarity_pairs")
    try:
        run_meta_df = load_delta("semantic_model_similarity_run")
    except AnalysisException:
        run_meta_df = pd.DataFrame()

    report_rows, report_scans = [], []
    report_load_status = "available"
    try:
        report_rows = load_delta("semantic_model_report_dependencies").to_dict("records")
        report_scans = load_delta("semantic_model_report_scan").to_dict("records")
    except AnalysisException as error:
        error_class = getattr(error, "getErrorClass", lambda: "")()
        report_load_status = "not_collected" if error_class in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST") else "unavailable"

    if models_df.empty:
        raise ValueError(
            "No rows in semantic_models in the attached lakehouse. "
            "Run 001_semantic_model_tom_catalog first."
        )

    def norm(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return re.sub(r"\s+", " ", str(value)).strip().casefold()

    def norm_dax(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        text = re.sub(r"/\*.*?\*/", " ", str(value), flags=re.DOTALL)
        text = re.sub(r"//.*", " ", text)
        return re.sub(r"\s+", " ", text).strip().casefold()

    def disp(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return str(value)

    def rounded(value):
        try:
            return round(float(value), 4)
        except (TypeError, ValueError):
            return None

    signatures = {}
    for _, row in models_df.iterrows():
        model_id = str(row["model_id"])
        signatures[model_id] = {
            "model_id": model_id,
            "workspace_id": disp(row.get("workspace_id")),
            "workspace_name": disp(row.get("workspace_name")),
            "model_name": disp(row.get("model_name")) or model_id,
        }

    report_payload = build_report_dependency_payload(signatures, report_rows, report_scans, report_load_status)

    if not run_meta_df.empty:
        meta = run_meta_df.iloc[0]
        duplicate_threshold = float(meta["duplicate_threshold"])
        similar_threshold = float(meta["similar_threshold"])
        containment_threshold = float(meta["containment_threshold"])
    else:
        duplicate_threshold, similar_threshold, containment_threshold = 0.95, 0.70, 0.95

    # Shared DAX pool: repeated expressions are serialized once.
    dax_pool = []
    dax_index = {}

    def dax_id(text):
        if not text:
            return -1
        idx = dax_index.get(text)
        if idx is None:
            idx = len(dax_pool)
            dax_pool.append(text)
            dax_index[text] = idx
        return idx

    def dax_hash(text):
        normalized = norm_dax(text)
        return hashlib.md5(normalized.encode("utf-8")).hexdigest()[:12] if normalized else ""

    # Inventories include every catalog model so direct Compare and the map cover the full comparable estate.
    inventories = {
        model_id: {
            "tables": {},
            "columns": {},
            "measures": {},
            "relationships": {},
            "datasources": {},
        }
        for model_id in signatures
    }

    for _, row in tables_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = norm(row["table_name"])
            if key:
                inventories[model_id]["tables"].setdefault(key, disp(row["table_name"]))

    for _, row in columns_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            table_key, column_key = norm(row["table_name"]), norm(row["column_name"])
            key = table_key + "." + column_key
            if column_key:
                inventories[model_id]["columns"].setdefault(
                    key,
                    {
                        "table": disp(row["table_name"]),
                        "tableKey": table_key,
                        "name": disp(row["column_name"]),
                    },
                )

    for _, row in measures_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = norm(row["measure_name"])
            expression = disp(row.get("expression"))
            if key:
                inventories[model_id]["measures"].setdefault(
                    key,
                    {
                        "name": disp(row["measure_name"]),
                        "daxId": dax_id(expression),
                        "daxHash": dax_hash(expression),
                    },
                )

    for _, row in relationships_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            key = (
                f"{norm(row['from_table'])}.{norm(row['from_column'])}"
                f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
            )
            inventories[model_id]["relationships"].setdefault(
                key,
                {
                    "from": f"{disp(row['from_table'])}[{disp(row['from_column'])}]",
                    "to": f"{disp(row['to_table'])}[{disp(row['to_column'])}]",
                },
            )

    for _, row in datasources_df.iterrows():
        model_id = str(row["model_id"])
        if model_id in inventories:
            connection = (
                row.get("connection_string")
                or row.get("connection_details")
                or row.get("datasource_name")
            )
            key = norm(connection)
            if key:
                inventories[model_id]["datasources"].setdefault(key, disp(connection))

    models_payload = {}
    for model_id, identity in signatures.items():
        inv = inventories[model_id]
        models_payload[model_id] = {
            "name": identity["model_name"],
            "workspace": identity["workspace_name"],
            "tables": [{"key": k, "name": v} for k, v in inv["tables"].items()],
            "columns": [
                {
                    "key": k,
                    "table": v["table"],
                    "tableKey": v["tableKey"],
                    "name": v["name"],
                }
                for k, v in inv["columns"].items()
            ],
            "measures": [
                {
                    "key": k,
                    "name": v["name"],
                    "daxId": v["daxId"],
                    "daxHash": v["daxHash"],
                }
                for k, v in inv["measures"].items()
            ],
            "relationships": [
                {"key": k, "from": v["from"], "to": v["to"]}
                for k, v in inv["relationships"].items()
            ],
            "datasources": [{"key": k, "name": v} for k, v in inv["datasources"].items()],
        }

    def pair_object(row):
        id_a, id_b = str(row["model_id_a"]), str(row["model_id_b"])
        sig_a, sig_b = signatures.get(id_a, {}), signatures.get(id_b, {})
        coverage_a_in_b = rounded(row.get("model_a_in_model_b"))
        coverage_b_in_a = rounded(row.get("model_b_in_model_a"))
        relationship = disp(row.get("containment_relationship"))

        contained_id = containing_id = ""
        contained_coverage = None
        if relationship == "model_a_contains_model_b":
            contained_id, containing_id, contained_coverage = id_b, id_a, coverage_b_in_a
        elif relationship == "model_b_contains_model_a":
            contained_id, containing_id, contained_coverage = id_a, id_b, coverage_a_in_b
        elif relationship == "equivalent":
            contained_coverage = max(coverage_a_in_b or 0, coverage_b_in_a or 0)
        elif (coverage_a_in_b or 0) >= (coverage_b_in_a or 0):
            contained_id, containing_id, contained_coverage = id_a, id_b, coverage_a_in_b
        else:
            contained_id, containing_id, contained_coverage = id_b, id_a, coverage_b_in_a

        return {
            "idA": id_a,
            "idB": id_b,
            "modelA": sig_a.get("model_name") or disp(row.get("model_a")) or id_a,
            "workspaceA": sig_a.get("workspace_name") or disp(row.get("workspace_a")),
            "modelB": sig_b.get("model_name") or disp(row.get("model_b")) or id_b,
            "workspaceB": sig_b.get("workspace_name") or disp(row.get("workspace_b")),
            "scored": True,
            "composite": rounded(row.get("composite_score")),
            "containment": rounded(row.get("containment_score")),
            "relationship": relationship,
            "containedId": contained_id,
            "containingId": containing_id,
            "containedCoverage": contained_coverage,
            "aInB": coverage_a_in_b,
            "bInA": coverage_b_in_a,
            "crossWorkspace": bool(row.get("cross_workspace", False)),
            "sameName": bool(row.get("same_model_name", False)),
            "jaccard": {
                "tables": rounded(row.get("jaccard_tables")),
                "columns": rounded(row.get("jaccard_columns")),
                "measures": rounded(row.get("jaccard_measure_names")),
                "relationships": rounded(row.get("jaccard_relationships")),
                "datasources": rounded(row.get("jaccard_datasources")),
            },
            "daxCosine": rounded(row.get("dax_embedding_cosine")),
        }

    all_pairs_payload = [pair_object(row) for _, row in pairs_df.iterrows()]
    model_list = sorted(
        [
            {
                "id": model_id,
                "name": model["name"],
                "workspace": model["workspace"],
            }
            for model_id, model in models_payload.items()
        ],
        key=lambda item: (item["name"].casefold(), item["workspace"].casefold()),
    )

    default_a = default_b = ""
    if all_pairs_payload:
        best_pair = max(all_pairs_payload, key=lambda p: p["composite"] or 0)
        default_a, default_b = best_pair["idA"], best_pair["idB"]
    elif len(model_list) >= 2:
        default_a, default_b = model_list[0]["id"], model_list[1]["id"]

    app_data = {
        "generatedAt": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        "summary": {"models": len(model_list)},
        "thresholds": {
            "duplicate": rounded(duplicate_threshold),
            "similar": rounded(similar_threshold),
            "containment": rounded(containment_threshold),
        },
        "allPairs": all_pairs_payload,
        "models": models_payload,
        "modelList": model_list,
        "daxPool": dax_pool,
        "defaultCompare": {"a": default_a, "b": default_b},
        "reportDependencies": report_payload,
    }

    app_template = r"""
<script>
  (() => {
    const param = new URLSearchParams(window.location.search).get("scoutTheme");
    const theme =
      param || (window.matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light");
    document.documentElement.setAttribute("data-theme", theme);
  })();
</script>
<style>
:root {
  color-scheme: light;
  --cp-bg: #f7f4ef;
  --cp-bg-elevated: #fcfbf8;
  --cp-surface: #ffffff;
  --cp-surface-soft: #f5f5f5;
  --cp-border: #dedede;
  --cp-border-strong: #919191;
  --cp-text: #242424;
  --cp-text-muted: #5c5c5c;
  --cp-text-soft: #6f6f6f;
  --cp-accent: #b11f4b;
  --cp-accent-hover: #9a1a41;
  --cp-accent-soft: rgba(177, 31, 75, 0.08);
  --cp-accent-fg: #ffffff;
  --cp-success: #16a34a;
  --cp-danger: #dc2626;
  --cp-warning: #f59e0b;
  --cp-link: #0078d4;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.12);
  --cp-overlay: rgba(255, 255, 255, 0.8);
  --cp-panel: rgba(255, 255, 255, 0.86);
  --cp-panel-strong: rgba(255, 255, 255, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.55);
  --cp-highlight: rgba(177, 31, 75, 0.12);
}
html[data-theme="dark"] {
  color-scheme: dark;
  --cp-bg: #3d3b3a;
  --cp-bg-elevated: #343231;
  --cp-surface: #292929;
  --cp-surface-soft: #2e2e2e;
  --cp-border: #474747;
  --cp-border-strong: #5f5f5f;
  --cp-text: #dedede;
  --cp-text-muted: #919191;
  --cp-text-soft: #b0b0b0;
  --cp-accent: #fd8ea1;
  --cp-accent-hover: #fb7b91;
  --cp-accent-soft: rgba(253, 142, 161, 0.14);
  --cp-accent-fg: #1a1a1a;
  --cp-success: #4ade80;
  --cp-danger: #f87171;
  --cp-warning: #fbbf24;
  --cp-link: #4da6ff;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.32);
  --cp-overlay: rgba(41, 41, 41, 0.88);
  --cp-panel: rgba(41, 41, 41, 0.72);
  --cp-panel-strong: rgba(41, 41, 41, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.04);
  --cp-highlight: rgba(253, 142, 161, 0.12);
}
#sms-app {
  background: var(--cp-bg);
  color: var(--cp-text);
  font-family: "Segoe UI", Aptos, Calibri, -apple-system, BlinkMacSystemFont, sans-serif;
  max-width: 1180px;
  margin: 8px auto;
  border: 1px solid var(--cp-border);
  border-radius: 16px;
  overflow: hidden;
}
#sms-app * { box-sizing: border-box; letter-spacing: 0; }
#sms-app button, #sms-app input, #sms-app select { font: inherit; max-width: 100%; }
#sms-app button { overflow-wrap: anywhere; }
#sms-app button, #sms-app select, #sms-app input[type="text"], #sms-app input[type="number"] { border-radius: 0.625rem; }
#sms-app button:focus-visible, #sms-app input:focus-visible, #sms-app select:focus-visible, #sms-app a:focus-visible, #sms-app summary:focus-visible, #sms-app [tabindex]:focus-visible {
  outline: 3px solid var(--cp-accent);
  outline-offset: 2px;
}
#sms-app .app-head { display: flex; justify-content: space-between; gap: 16px; padding: 24px 28px 16px; }
#sms-app .app-head > div { min-width: 0; }
#sms-app .eyebrow { color: var(--cp-text-muted); font-size: 12px; font-weight: 700; text-transform: uppercase; }
#sms-app h1 { font-size: 23px; margin: 4px 0; }
#sms-app .subtitle, #sms-app .muted { color: var(--cp-text-muted); }
#sms-app .subtitle { font-size: 13px; }
#sms-app .head-controls { display: flex; flex-direction: column; align-items: flex-end; gap: 8px; }
#sms-app .control-row, #sms-app .toolbar, #sms-app .filter-row, #sms-app .compare-controls, #sms-app .legend { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
#sms-app .generated { color: var(--cp-text-muted); font-size: 11px; }
#sms-app .button, #sms-app .tab, #sms-app .filter, #sms-app .theme-button, #sms-app .disclosure {
  border: 1px solid var(--cp-border);
  background: var(--cp-surface);
  color: var(--cp-text);
  cursor: pointer;
  padding: 8px 12px;
}
#sms-app .button:hover, #sms-app .tab:hover, #sms-app .filter:hover, #sms-app .theme-button:hover, #sms-app .disclosure:hover { border-color: var(--cp-border-strong); }
#sms-app .button.primary { background: var(--cp-accent); border-color: var(--cp-accent); color: var(--cp-accent-fg); font-weight: 700; }
#sms-app .button.primary:hover { background: var(--cp-accent-hover); }
#sms-app .theme-button.active, #sms-app .filter.active { background: var(--cp-accent-soft); border-color: var(--cp-accent); color: var(--cp-accent); font-weight: 700; }
#sms-app .tabs { display: flex; gap: 4px; padding: 4px; margin: 0 28px; background: var(--cp-surface-soft); border-radius: 0.625rem; overflow-x: auto; }
#sms-app .tab { flex: 0 0 auto; border-color: var(--cp-surface-soft); background: var(--cp-surface-soft); font-weight: 700; }
#sms-app .tab[aria-selected="true"] { background: var(--cp-surface); border-color: var(--cp-border); color: var(--cp-accent); }
#sms-app .settings { margin: 8px 28px 0; }
#sms-app .settings-panel { background: var(--cp-surface); border: 1px solid var(--cp-border); border-radius: 16px; padding: 16px; box-shadow: 0 0 2px var(--cp-border), 0 1px 2px var(--cp-border); }
#sms-app .settings-grid { display: grid; gap: 12px; }
#sms-app .setting { display: grid; grid-template-columns: minmax(180px, 1fr) 2fr 84px; gap: 12px; align-items: center; }
#sms-app .setting small { display: block; color: var(--cp-text-muted); }
#sms-app input[type="range"] { width: 100%; accent-color: var(--cp-accent); }
#sms-app input[type="number"], #sms-app input[type="text"], #sms-app select { background: var(--cp-surface); border: 1px solid var(--cp-border); color: var(--cp-text); padding: 8px 10px; }
#sms-app .settings-actions { display: flex; justify-content: space-between; align-items: center; gap: 12px; margin-top: 12px; padding-top: 12px; border-top: 1px solid var(--cp-border); }
#sms-app .view { padding: 22px 28px 28px; }
#sms-app .view h2 { font-size: 19px; margin: 0 0 6px; }
#sms-app .intro { color: var(--cp-text-muted); font-size: 13px; margin: 0 0 18px; line-height: 1.5; }
#sms-app .stats { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 12px; margin-bottom: 18px; }
#sms-app .card, #sms-app .stat, #sms-app .empty, #sms-app .compare-summary, #sms-app .map-wrap {
  background: var(--cp-surface);
  border: 1px solid var(--cp-border);
  border-radius: 16px;
  box-shadow: 0 0 2px var(--cp-border), 0 1px 2px var(--cp-border);
}
#sms-app .stat { padding: 16px; }
#sms-app .stat strong { display: block; font-size: 29px; }
#sms-app .stat span { color: var(--cp-text-muted); font-size: 12px; }
#sms-app .toolbar { justify-content: space-between; margin: 16px 0 12px; }
#sms-app .search, #sms-app .report-search { flex: 1 1 220px; min-width: 0; }
#sms-app .filter { font-size: 12px; font-weight: 700; }
#sms-app .queue { display: grid; gap: 12px; }
#sms-app .candidate { padding: 16px; }
#sms-app .candidate-main { display: grid; grid-template-columns: minmax(0, 1fr) minmax(160px, 220px); gap: 24px; align-items: start; }
#sms-app .relationship { color: var(--cp-accent); font-size: 12px; font-weight: 800; text-transform: uppercase; }
#sms-app .finding { font-size: 16px; line-height: 1.4; margin: 6px 0 12px; overflow-wrap: anywhere; }
#sms-app .identity-row { display: flex; align-items: stretch; gap: 10px; margin: 8px 0; }
#sms-app .identity { flex: 1 1 0; min-width: 0; }
#sms-app .model-name { font-weight: 700; overflow-wrap: anywhere; }
#sms-app .workspace { color: var(--cp-text-muted); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .relation-word { align-self: center; color: var(--cp-text-muted); font-size: 12px; font-weight: 700; }
#sms-app .plain { color: var(--cp-text-soft); font-size: 13px; line-height: 1.5; overflow-wrap: anywhere; }
#sms-app .metadata { color: var(--cp-text-muted); font-size: 11px; margin-top: 6px; overflow-wrap: anywhere; }
#sms-app .score { text-align: right; min-width: 0; overflow-wrap: anywhere; }
#sms-app .score strong { display: block; font-size: 24px; font-variant-numeric: tabular-nums; }
#sms-app .score span { display: block; color: var(--cp-text-muted); font-size: 12px; }
#sms-app .score .direction-score + .direction-score { margin-top: 12px; }
#sms-app .secondary-score { margin-top: 12px; }
#sms-app .secondary-score strong { display: block; color: var(--cp-text); font-size: 13px; }
#sms-app .caution { margin: 12px 0 0; color: var(--cp-text-soft); font-size: 12px; line-height: 1.5; }
#sms-app .candidate-actions { display: flex; gap: 8px; justify-content: flex-end; flex-wrap: wrap; margin-top: 12px; }
#sms-app .evidence { border-top: 1px solid var(--cp-border); margin-top: 12px; padding-top: 12px; }
#sms-app .score-section + .score-section { border-top: 1px solid var(--cp-border); padding-top: 16px; margin-top: 16px; }
#sms-app .score-section h4 { font-size: 14px; margin: 0 0 6px; }
#sms-app .score-section p { margin: 6px 0; }
#sms-app .coverage-list { display: block; width: 100%; max-width: none; margin: 8px 0; padding: 0; }
#sms-app .coverage-list > div { display: grid; grid-template-columns: minmax(0, 1fr) max-content; align-items: baseline; gap: 16px; padding: 8px 0; font-size: 12px; line-height: 1.5; }
#sms-app .coverage-list dt, #sms-app .coverage-list dd { display: block; position: static; float: none; clear: none; width: auto; max-width: none; min-width: 0; margin: 0; padding: 0; }
#sms-app .coverage-list dt { font-weight: 400; text-align: left; overflow-wrap: anywhere; }
#sms-app .coverage-list dd { justify-self: end; text-align: right; white-space: nowrap; font-weight: 700; font-variant-numeric: tabular-nums; }
#sms-app .evidence-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); grid-auto-rows: 1fr; gap: 8px; width: 100%; max-width: none; margin: 12px 0; padding: 0; }
#sms-app .evidence-item { display: flex; flex-direction: column; min-width: 0; min-height: 86px; background: var(--cp-surface-soft); border-radius: 8px; padding: 10px; overflow-wrap: anywhere; }
#sms-app .evidence-item dt, #sms-app .evidence-item dd { display: block; position: static; float: none; clear: none; width: auto; max-width: none; min-width: 0; margin: 0; padding: 0; text-align: left; }
#sms-app .evidence-item dt { flex: 1; font-size: 11px; font-weight: 400; line-height: 1.4; }
#sms-app .evidence-item dd { margin-top: 4px; }
#sms-app .metric-value { display: block; font-size: 14px; line-height: 1.3; font-weight: 700; font-variant-numeric: tabular-nums; white-space: nowrap; }
#sms-app .metric-meter { display: block; width: 100%; height: 6px; margin-top: 6px; background: var(--cp-border); border-radius: 3px; overflow: hidden; }
#sms-app .metric-fill { display: block; height: 100%; background: var(--cp-accent); border-radius: inherit; }
#sms-app .metric-meter.unavailable { background: var(--cp-surface); border: 1px dashed var(--cp-border-strong); }
#sms-app .metric-scale { display: flex; justify-content: space-between; margin-top: 4px; color: var(--cp-text-muted); font-size: 9px; line-height: 1.2; font-weight: 400; }
#sms-app .empty { padding: 28px; text-align: center; color: var(--cp-text-muted); }
#sms-app .method { margin-top: 8px; font-size: 12px; color: var(--cp-text-soft); line-height: 1.5; }
#sms-app .method summary { cursor: pointer; font-weight: 600; }
#sms-app .groups { display: grid; gap: 12px; }
#sms-app .group { padding: 16px; }
#sms-app .group-head { display: flex; justify-content: space-between; gap: 12px; align-items: baseline; flex-wrap: wrap; }
#sms-app .group-head h3 { margin: 0; font-size: 16px; overflow-wrap: anywhere; }
#sms-app .group-members { display: grid; gap: 4px; list-style: none; padding: 0; margin: 12px 0; }
#sms-app .group-members li { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); gap: 8px; border-top: 1px solid var(--cp-border); padding-top: 6px; }
#sms-app .group-pickers { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr) auto; gap: 8px; align-items: end; }
#sms-app label { color: var(--cp-text-soft); font-size: 12px; font-weight: 600; min-width: 0; }
#sms-app label select { display: block; width: 100%; margin-top: 4px; }
#sms-app .compare-controls { margin-bottom: 14px; }
#sms-app .compare-controls label { flex: 1 1 230px; min-width: 0; }
#sms-app select { max-width: 100%; }
#sms-app .compare-summary { padding: 16px; margin-bottom: 12px; }
#sms-app .summary-grid { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin-top: 12px; }
#sms-app .summary-item { min-width: 0; overflow-wrap: anywhere; }
#sms-app .summary-item > span { display: block; color: var(--cp-text-muted); font-size: 12px; }
#sms-app .summary-item > strong { font-size: 16px; }
#sms-app .diff-section { margin-bottom: 10px; }
#sms-app .section-button { width: 100%; display: flex; justify-content: space-between; align-items: center; gap: 12px; text-align: left; background: var(--cp-surface); border: 1px solid var(--cp-border); color: var(--cp-text); border-radius: 16px; padding: 13px 16px; cursor: pointer; font-weight: 700; }
#sms-app .section-button > span { min-width: 0; overflow-wrap: anywhere; }
#sms-app .section-body { background: var(--cp-surface); border: 1px solid var(--cp-border); border-top: 0; padding: 12px 16px; border-radius: 0 0 16px 16px; }
#sms-app .diff-row { display: flex; gap: 8px; padding: 5px 7px; border-radius: 0.625rem; font-size: 12px; }
#sms-app .diff-row > span { min-width: 0; overflow-wrap: anywhere; }
#sms-app .diff-row.onlyA, #sms-app .diff-row.onlyB, #sms-app .diff-row.changed { background: var(--cp-accent-soft); }
#sms-app .status { color: var(--cp-accent); flex: 0 0 40%; font-weight: 700; }
#sms-app .dax { font-family: Consolas, "Courier New", Courier, monospace; white-space: pre-wrap; overflow-wrap: anywhere; background: var(--cp-surface-soft); border: 1px solid var(--cp-border); border-radius: 0.625rem; padding: 8px; margin: 4px 0 10px 100px; }
#sms-app .map-note { border-left: 4px solid var(--cp-warning); padding: 8px 12px; background: var(--cp-surface-soft); margin-bottom: 12px; font-size: 12px; }
#sms-app .map-wrap { overflow-x: auto; padding: 16px; }
#sms-app .matrix { border-collapse: separate; border-spacing: 3px; min-width: max-content; }
#sms-app .matrix th { max-width: 150px; color: var(--cp-text-muted); font-size: 10px; font-weight: 700; text-align: left; overflow-wrap: anywhere; }
#sms-app .matrix th.col { writing-mode: vertical-rl; transform: rotate(180deg); height: 180px; text-align: right; }
#sms-app .matrix tbody tr { background: var(--cp-surface); }
#sms-app .matrix tbody tr:nth-child(even) { background: var(--cp-surface-soft); }
#sms-app .matrix tbody td, #sms-app .matrix tbody th { background: inherit; }
#sms-app .matrix-cell { width: 44px; height: 36px; padding: 0; border: 1px solid var(--cp-border); border-radius: 0.625rem; font-size: 10px; font-weight: 800; }
#sms-app button.matrix-cell { cursor: pointer; }
#sms-app button.matrix-cell:hover { border-color: var(--cp-accent); outline: 0; box-shadow: 0 0 0 2px var(--cp-surface), 0 0 0 5px var(--cp-accent); position: relative; z-index: 1; }
#sms-app button.matrix-cell:focus-visible { border-color: var(--cp-accent); outline: 0; box-shadow: 0 0 0 2px var(--cp-surface), 0 0 0 5px var(--cp-accent); position: relative; z-index: 1; }
#sms-app .matrix-cell.duplicate { background: var(--cp-accent); color: var(--cp-accent-fg); }
#sms-app .matrix-cell.high { background: var(--cp-highlight); color: var(--cp-text); }
#sms-app .matrix-cell.low, #sms-app .matrix-cell.unavailable { background: var(--cp-surface-soft); color: var(--cp-text-muted); }
#sms-app .matrix-cell.unscored { background: var(--cp-bg-elevated); color: var(--cp-text-muted); }
#sms-app .matrix-cell.diagonal { display: inline-flex; justify-content: center; align-items: center; background: var(--cp-border); color: var(--cp-text); }
#sms-app .legend { margin-top: 12px; font-size: 11px; color: var(--cp-text-muted); }
#sms-app .legend-key { display: inline-block; width: 12px; height: 12px; border: 1px solid var(--cp-border); border-radius: 4px; vertical-align: middle; }
#sms-app .legend-key.duplicate { background: var(--cp-accent); }
#sms-app .legend-key.high { background: var(--cp-highlight); }
#sms-app .legend-key.low { background: var(--cp-surface-soft); }
#sms-app .legend-key.unscored { background: var(--cp-bg-elevated); }
#sms-app .legend-key.unavailable, #sms-app .matrix-cell.unavailable { border-style: dashed; }
#sms-app .report-count { color: var(--cp-text-soft); font-size: 11px; margin-top: 4px; }
#sms-app .report-banner { padding: 10px 12px; margin: 12px 0; border-left: 4px solid var(--cp-warning); background: var(--cp-surface-soft); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .report-banner.complete { border-color: var(--cp-success); }
#sms-app .report-model { border-top: 1px solid var(--cp-border); padding: 14px 0; }
#sms-app .report-model summary { display: flex; flex-wrap: wrap; gap: 8px; align-items: baseline; cursor: pointer; }
#sms-app .report-model summary .report-count { margin-left: auto; }
#sms-app .report-list { list-style: none; margin: 8px 0 0; padding: 0; }
#sms-app .report-list li { display: grid; grid-template-columns: minmax(0, 1fr) minmax(100px, .6fr) auto; gap: 10px; align-items: baseline; padding: 9px 0; border-top: 1px solid var(--cp-border); font-size: 12px; overflow-wrap: anywhere; }
#sms-app .report-list a { color: var(--cp-link); text-underline-offset: 3px; overflow-wrap: anywhere; }
#sms-app .report-list .report-kind { color: var(--cp-text-muted); font-size: 11px; }
#sms-app .report-compare { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 20px; }
#sms-app .report-compare h3 { font-size: 14px; margin: 0 0 6px; overflow-wrap: anywhere; }
#sms-app .report-compare .report-list li { grid-template-columns: 1fr; gap: 4px; }
#sms-app .scan-details { border-top: 1px solid var(--cp-border); padding-top: 12px; margin-top: 12px; font-size: 12px; }
#sms-app .scan-details summary { cursor: pointer; }
#sms-app .scan-row { display: grid; grid-template-columns: minmax(0, 1fr) auto auto; gap: 12px; padding: 7px 0; border-bottom: 1px solid var(--cp-border); overflow-wrap: anywhere; }
#sms-app .foot { color: var(--cp-text-muted); font-size: 11px; padding: 0 28px 22px; line-height: 1.5; }
#sms-app [hidden] { display: none !important; }
@media (max-width: 720px) {
  #sms-app .app-head { flex-direction: column; padding: 20px 16px 12px; }
  #sms-app .head-controls { align-items: flex-start; }
  #sms-app .tabs, #sms-app .settings { margin-left: 16px; margin-right: 16px; }
  #sms-app .view { padding: 18px 16px 22px; }
  #sms-app .stats, #sms-app .summary-grid, #sms-app .report-compare { grid-template-columns: 1fr; }
  #sms-app .candidate-main { grid-template-columns: 1fr; gap: 12px; }
#sms-app .score { text-align: left; }
  #sms-app .identity-row { flex-direction: column; }
  #sms-app .relation-word { align-self: flex-start; }
  #sms-app .setting, #sms-app .group-pickers { grid-template-columns: 1fr; }
  #sms-app .group-members li, #sms-app .report-list li, #sms-app .scan-row { grid-template-columns: 1fr; gap: 4px; }
  #sms-app .report-model summary { flex-direction: column; gap: 4px; }
  #sms-app .report-model summary .report-count { margin-left: 0; }
  #sms-app .candidate-actions, #sms-app .settings-actions { flex-wrap: wrap; justify-content: flex-start; }
  #sms-app .evidence-grid { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  #sms-app .section-button { flex-wrap: wrap; }
  #sms-app .dax { margin-left: 0; }
}
</style>
<div id="sms-app"></div>
<script>
(function(){
  var DATA = __APP_DATA__;
  var root = document.getElementById('sms-app');
  if(!root){ return; }
  var MODELS = DATA.models || {}, MLIST = DATA.modelList || [], DAXPOOL = DATA.daxPool || [];
  var REPORTS = DATA.reportDependencies || {status:'not_collected',byModel:{}};
  var DEFAULTS = DATA.thresholds || {duplicate:.95, similar:.70, containment:.95};
  var LABELS = {duplicate:'Possible duplicates',containment:'Model coverage',overlap:'Shared structure'};
  var COVERAGE_MEANING = 'A weighted match across cataloged model definitions, not a percentage of objects or data values.';
  var SCORE_DIFFERENCE = 'Overall similarity compares both complete models. Extra content can lower this score even when one model has high coverage within the other.';
  var REVIEW_CAUTION = 'This is a review candidate, not confirmation that one model can replace the other.';
  var initialTheme = document.documentElement.getAttribute('data-theme') || 'light';
  var state = {
    tab:'review', filter:'all', search:'', theme:initialTheme, settingsOpen:false,
    thresholds:{duplicate:DEFAULTS.duplicate, similar:DEFAULTS.similar, containment:DEFAULTS.containment},
    draft:{duplicate:DEFAULTS.duplicate, similar:DEFAULTS.similar, containment:DEFAULTS.containment},
    cmpA:(DATA.defaultCompare||{}).a || '', cmpB:(DATA.defaultCompare||{}).b || '',
    cmpDiffOnly:false, openSections:{}, groupSelections:{}, reportSearch:'', reportsOnly:false
  };
  try {
    var savedTheme=localStorage.getItem('sms-theme');
    if(savedTheme==='light'||savedTheme==='dark'){ state.theme=savedTheme; }
    var savedThresholds=JSON.parse(localStorage.getItem('sms-thresholds')||'null');
    if(savedThresholds){ ['duplicate','similar','containment'].forEach(function(key){ if(typeof savedThresholds[key]==='number'&&Number.isFinite(savedThresholds[key])){ state.thresholds[key]=Math.max(0,Math.min(1,savedThresholds[key])); } }); state.draft=Object.assign({},state.thresholds); }
  } catch(e){}
  if(!state.cmpA && MLIST[0]){ state.cmpA=MLIST[0].id; }
  if(!state.cmpB && MLIST[1]){ state.cmpB=MLIST[1].id; }

  function esc(s){ return String(s==null?'':s).replace(/[&<>"']/g,function(c){ return {'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]; }); }
  function validScore(value){ return typeof value==='number'&&Number.isFinite(value)&&value>=0&&value<=1; }
  function score(value){ return validScore(value)?(Math.round(value*1000)/10).toFixed(1)+'%':'Unavailable'; }
  function model(id){ return MODELS[id] || {name:id||'Model unavailable',workspace:''}; }
  function identity(id){ var entry=model(id); return '<div class="identity"><div class="model-name">'+esc(entry.name)+'</div><div class="workspace">Workspace: '+esc(entry.workspace||'Unavailable')+'</div><div class="report-count">'+esc(reportCountText(id))+'</div></div>'; }
  function pairModelLabel(id,otherId){
    var entry=model(id),other=model(otherId),label=entry.name;
    if(String(entry.name).toLowerCase()===String(other.name).toLowerCase()){
      label+=' (Workspace: '+(entry.workspace||'Unavailable')+')';
      if(String(entry.workspace).toLowerCase()===String(other.workspace).toLowerCase()){label+=' ['+id+']';}
    }
    return label;
  }
  function pairKey(a,b){ return [String(a),String(b)].sort().join('|'); }
  function pairMap(){ var out={}; (DATA.allPairs||[]).forEach(function(p){ out[pairKey(p.idA,p.idB)]=p; }); return out; }
  function daxText(item){ return item&&item.daxId>=0 ? (DAXPOOL[item.daxId]||'') : ''; }
  function tierAt(value){ return !validScore(value)?'distinct':(value>=state.thresholds.duplicate?'duplicate':(value>=state.thresholds.similar?'similar':'distinct')); }
  function coverageContext(pair){
    var directions=pair?[
      {sourceId:pair.idA,targetId:pair.idB,value:pair.aInB},
      {sourceId:pair.idB,targetId:pair.idA,value:pair.bInA}
    ]:[];
    var available=directions.filter(function(direction){return direction.sourceId&&direction.targetId&&validScore(direction.value);});
    var strongest=available.reduce(function(best,direction){return !best||direction.value>best.value?direction:best;},null);
    var matches=available.filter(function(direction){return direction.value>=state.thresholds.containment;});
    return {directions:directions,strongest:strongest,both:matches.length===2,qualifies:matches.length>0};
  }
  function directionLabel(direction){return pairModelLabel(direction.sourceId,direction.targetId)+' within '+pairModelLabel(direction.targetId,direction.sourceId);}
  function coverageRowsHTML(pair){
    if(!pair){return '<div class="plain">Not scored</div>';}
    return '<dl class="coverage-list">'+coverageContext(pair).directions.map(function(direction){return '<div><dt>Coverage score: '+esc(directionLabel(direction))+'</dt><dd>'+score(direction.value)+'</dd></div>';}).join('')+'</dl>';
  }

  function reportsAvailable(){return REPORTS.status==='complete'||REPORTS.status==='partial';}
  function reportsFor(modelId){return reportsAvailable()?((REPORTS.byModel||{})[modelId]||[]):[];}
  function reportCountText(modelId){
    if(!reportsAvailable()){return 'Report dependencies unknown';}
    var count=reportsFor(modelId).length;
    if(REPORTS.status==='partial'){
      return count?count+' known linked report'+(count===1?'':'s')+' (scan incomplete)':'No linked reports found (scan incomplete)';
    }
    return count+' linked report'+(count===1?'':'s')+' in scanned scope';
  }
  function reportStatusHTML(){
    var labels={complete:'Report scan complete within scanned scope',partial:'Partial report scan: dependencies may be missing',not_collected:'Report dependencies were not collected',unavailable:'Report dependency data is unavailable',inconsistent:'Report snapshots do not agree; dependencies are unknown'};
    var scope=REPORTS.scope||'Unknown scope';
    if(scope==='visible_workspaces'){scope='Workspaces visible to the scanning identity';}
    else if(scope.indexOf('workspace_filter:')===0){scope='Workspace filter: '+scope.slice('workspace_filter:'.length);}
    var detail=reportsAvailable()?'Report snapshot: '+esc(REPORTS.scannedAt||'Time unavailable')+' · '+esc(REPORTS.workspaceCount||0)+' workspaces · '+esc(scope):'Report dependencies unknown';
    return '<div class="report-banner '+(REPORTS.status==='complete'?'complete':'')+'" role="status"><strong>'+esc(labels[REPORTS.status]||'Report dependencies unknown')+'</strong><div class="metadata">'+detail+'</div>'+(reportsAvailable()?'<div class="plain">Direct report links observed in the scanned scope, not usage. Zero does not prove a model is unused.</div>':'')+'</div>';
  }
  function reportListHTML(reports){
    if(!reports.length){return '';}
    return '<ul class="report-list">'+reports.map(function(report){
      var label=esc(report.name),url;
      try{url=new URL(report.url);if(url.protocol==='https:'&&url.hostname==='app.powerbi.com'&&!url.port&&!url.username&&!url.password){label='<a href="'+esc(url.href)+'" target="_blank" rel="noopener noreferrer">'+label+'</a>';}}catch(error){}
      return '<li><span>'+label+'</span><span class="workspace">Workspace: '+esc(report.workspace)+'</span><span class="report-kind">'+esc(report.reason||(report.crossWorkspace?'Cross-workspace':'Same workspace'))+'</span></li>';
    }).join('')+'</ul>';
  }
  function reportCompareHTML(){
    var content=reportStatusHTML()+'<div class="report-compare">'+[state.cmpA,state.cmpB].map(function(modelId){return '<div><h3>'+esc(model(modelId).name)+'</h3><div class="workspace">Workspace: '+esc(model(modelId).workspace||'Unavailable')+'</div><div class="report-count">'+esc(reportCountText(modelId))+'</div>'+reportListHTML(reportsFor(modelId))+'</div>';}).join('')+'</div>';
    var summary=REPORTS.status==='complete'?'Observed direct links':(REPORTS.status==='partial'?'Scan incomplete':'Dependencies unknown');
    return sectionHTML('reports','Dependent reports',summary,[content]);
  }
  function reportRowsHTML(){
    var query=state.reportSearch.trim().toLowerCase();
    var filtered=MLIST.filter(function(entry){
      var reports=reportsFor(entry.id);if(state.reportsOnly&&!reports.length){return false;}
      return !query||[entry.name,entry.workspace].concat(reports.map(function(report){return report.name+' '+report.workspace;})).join(' ').toLowerCase().indexOf(query)>=0;
    });
    if(!filtered.length){return '<div class="muted">No matching models or reports in this inventory.</div>';}
    return filtered.map(function(entry){var reports=reportsFor(entry.id);return '<details class="report-model"'+(reports.length?' open':'')+'><summary><span class="model-name">'+esc(entry.name)+'</span><span class="workspace">Workspace: '+esc(entry.workspace)+'</span><span class="report-count">'+esc(reportCountText(entry.id))+'</span></summary>'+reportListHTML(reports)+'</details>';}).join('');
  }
  function reportsHTML(){
    var overview='<h2>Report dependencies</h2><p class="intro">Reports linked directly to cataloged models. These links provide impact context and do not change model scores.</p>'+reportStatusHTML();
    if(!reportsAvailable()){return overview;}
    var scans=(REPORTS.scans||[]).map(function(scan){return '<div class="scan-row"><span>'+esc(scan.workspace)+'</span><span>'+esc(scan.status)+(scan.errorType?' · '+esc(scan.errorType):'')+'</span><span>'+esc(scan.reportCount)+' reports found</span></div>';}).join('');
    return overview+'<div class="plain">'+esc(REPORTS.linkedReportCount||0)+' direct links to cataloged models · '+esc(REPORTS.reportCount||0)+' reports found'+(REPORTS.status==='partial'?' (scan incomplete)':' in scanned scope')+'</div><div class="toolbar"><input class="report-search" type="text" aria-label="Search report dependencies" placeholder="Search models, reports or workspaces" value="'+esc(state.reportSearch)+'"><label><input type="checkbox" data-reports-only'+(state.reportsOnly?' checked':'')+'> Models with linked reports found</label></div><div class="report-models">'+reportRowsHTML()+'</div>'+((REPORTS.otherReports||[]).length?'<h3>Reports without a resolved catalog link</h3>'+reportListHTML(REPORTS.otherReports):'')+'<details class="scan-details"><summary>Workspace scan status</summary>'+scans+'</details>';
  }

  function classify(pair){
    var tier=tierAt(pair.composite),context=coverageContext(pair);
    var coverage=context.strongest?context.strongest.value:(validScore(pair.containedCoverage)?pair.containedCoverage:pair.containment);
    if(tier==='duplicate'){ return {kind:'duplicate',label:LABELS.duplicate,rank:0,primary:pair.composite}; }
    if(validScore(coverage)&&coverage>=state.thresholds.containment){ return {kind:'containment',label:LABELS.containment,rank:1,primary:coverage}; }
    if(tier==='similar'){ return {kind:'overlap',label:LABELS.overlap,rank:2,primary:pair.composite}; }
    return null;
  }
  function candidates(){
    return (DATA.allPairs||[]).map(function(p){ var c=classify(p); return c?{pair:p,category:c}:null; }).filter(Boolean).sort(function(a,b){ return a.category.rank-b.category.rank || b.category.primary-a.category.primary; });
  }
  function duplicatePairs(){ return (DATA.allPairs||[]).filter(function(pair){ return tierAt(pair.composite)==='duplicate'; }); }
  function buildGroups(){
    var dup=duplicatePairs(), parent={};
    function find(x){ if(parent[x]!==x){ parent[x]=find(parent[x]); } return parent[x]; }
    function add(x){ if(!(x in parent)){ parent[x]=x; } }
    function join(a,b){ add(a);add(b);var ra=find(a),rb=find(b);if(ra!==rb){parent[rb]=ra;} }
    dup.forEach(function(p){join(p.idA,p.idB);});
    var grouped={}; Object.keys(parent).forEach(function(id){var r=find(id);(grouped[r]=grouped[r]||[]).push(id);});
    var pmap=pairMap(), groups=[];
    Object.keys(grouped).forEach(function(key){
      var members=grouped[key]; if(members.length<2){return;}
      var strongest=null;
      for(var i=0;i<members.length;i++){for(var j=i+1;j<members.length;j++){var p=pmap[pairKey(members[i],members[j])];if(p&&tierAt(p.composite)==='duplicate'&&(!strongest||(p.composite||0)>(strongest.composite||0))){strongest=p;}}}
      if(!strongest){return;}
      members.sort(function(a,b){return (model(a).name||'').localeCompare(model(b).name||'')||(model(a).workspace||'').localeCompare(model(b).workspace||'');});
      groups.push({members:members,strongest:strongest});
    });
    groups.sort(function(a,b){return b.members.length-a.members.length||(b.strongest.composite||0)-(a.strongest.composite||0);});
    return groups;
  }
  function relationshipText(pair,category){
    if(!pair){return 'This pair has no saved analysis score.';}
    if(!category){return 'This pair does not meet the current review thresholds, or its scores are unavailable.';}
    if(category.kind==='duplicate'){
      return pair.composite>=.95?'These models have very high overall similarity.':'These models meet the possible-duplicate threshold.';
    }
    if(category.kind==='containment'){
      var context=coverageContext(pair),direction=context.strongest;
      if(context.both){return 'Both models meet the coverage threshold.';}
      if(!direction){return 'Model coverage was flagged, but its direction is unavailable.';}
      var source=pairModelLabel(direction.sourceId,direction.targetId),target=pairModelLabel(direction.targetId,direction.sourceId);
      return direction.value>=.95?'Almost all of '+source+' is represented in '+target+'.':source+' has a '+score(direction.value)+' coverage score within '+target+'.';
    }
    return 'These models meet the shared-structure threshold, below the possible-duplicate threshold.';
  }
  function relevantSection(pair){
    var overlap=pair.jaccard||{};
    var values=[['tables',overlap.tables],['measures',Math.min(validScore(overlap.measures)?overlap.measures:1,validScore(pair.daxCosine)?pair.daxCosine:1)],['relationships',overlap.relationships],['datasources',overlap.datasources]];
    values.sort(function(first,second){return (validScore(first[1])?first[1]:1)-(validScore(second[1])?second[1]:1);}); return values[0][0];
  }

  function tabsHTML(){
    return '<div class="tabs" role="tablist" aria-label="Similarity results views">'+[
      ['review','Review'],['groups','Groups'],['map','Similarity map'],['compare','Compare']
    ].map(function(t){var selected=state.tab===t[0];return '<button class="tab" role="tab" id="tab-'+t[0]+'" aria-selected="'+selected+'" aria-controls="panel-'+t[0]+'" tabindex="'+(selected?'0':'-1')+'" data-tab="'+t[0]+'">'+t[1]+'</button>';}).join('')+'</div>';
  }
  function headerHTML(){
    return '<div class="app-head"><div><div class="eyebrow">Semantic Model Similarity</div><h1>Consolidation review</h1><div class="subtitle">Metadata matches identify candidates for review, not models proven safe to replace.</div></div><div class="head-controls"><div class="control-row"><button class="button" data-settings aria-expanded="'+state.settingsOpen+'" aria-controls="scoring-settings">Review thresholds</button><span class="muted">Theme:</span><button class="theme-button'+(state.theme==='light'?' active':'')+'" data-theme-set="light" aria-pressed="'+(state.theme==='light')+'">Light</button><button class="theme-button'+(state.theme==='dark'?' active':'')+'" data-theme-set="dark" aria-pressed="'+(state.theme==='dark')+'">Dark</button></div><div class="generated">View generated: '+esc(DATA.generatedAt)+'</div></div></div>';
  }
  function settingsHTML(){
    function row(key,label,hint){var value=Math.round(state.draft[key]*10000)/100;return '<div class="setting"><label for="range-'+key+'">'+label+'<small>'+hint+'</small></label><input id="range-'+key+'" type="range" min="0" max="100" step="0.01" value="'+value+'" data-draft-range="'+key+'" aria-label="'+label+' minimum score percent"><input type="number" min="0" max="100" step="0.01" value="'+value+'" data-draft-number="'+key+'" aria-label="'+label+' minimum score percent value"></div>'; }
    return '<div class="settings" id="scoring-settings"'+(state.settingsOpen?'':' hidden')+'><div class="settings-panel"><div class="settings-grid">'+row('duplicate',LABELS.duplicate,'Minimum overall similarity (%)')+row('similar',LABELS.overlap,'Minimum overall similarity (%), below possible duplicates')+row('containment',LABELS.containment,'Minimum coverage score (%) in at least one direction')+'</div><div class="settings-actions"><span class="muted">Cutoffs apply to existing scores. They do not recalculate scores or include pairs omitted from scoring.</span><div class="control-row"><button class="button" data-reset>Reset defaults</button><button class="button primary" data-apply>Apply thresholds</button></div></div></div></div>';
  }

  function metricHTML(label,value){
    var available=validScore(value),percent=available?Math.round(value*1000)/10:null;
    var meter=available?'<div class="metric-meter" role="meter" aria-label="'+esc(label)+'" aria-valuemin="0" aria-valuemax="100" aria-valuenow="'+percent+'" aria-valuetext="'+score(value)+'"><span class="metric-fill" style="width:'+percent+'%"></span></div>':'<div class="metric-meter unavailable" aria-hidden="true"></div>';
    return '<div class="evidence-item"><dt>'+esc(label)+'</dt><dd><span class="metric-value">'+score(value)+'</span>'+meter+'<span class="metric-scale" aria-hidden="true"><span>0%</span><span>100%</span></span></dd></div>';
  }
  function evidenceHTML(pair){
    if(!pair){return '<p class="plain">Not scored. Cataloged differences are still available; no similarity or coverage score was saved for this pair.</p>';}
    var overlap=pair.jaccard||{},hasMeasures=(model(pair.idA).measures||[]).length>0&&(model(pair.idB).measures||[]).length>0;
    var items=[['Table-name overlap',overlap.tables],['Column-name overlap',overlap.columns],['Measure-name overlap',overlap.measures],[hasMeasures?'DAX text similarity':'Model text similarity',pair.daxCosine],['Relationship-link overlap',overlap.relationships],['Source-definition overlap',overlap.datasources]];
    return '<section class="score-section"><h4>Model coverage</h4><p class="plain">How much of each model is represented in the other? Each direction checks a different source model.</p>'+coverageRowsHTML(pair)+'<p class="plain">'+COVERAGE_MEANING+'</p><details class="method"><summary>Coverage calculation</summary><p>For each source model, matching entries are divided by that source model\'s entries. The score combines six signals: table names, table-qualified column names, measure names, measure names with normalized formula text, relationship links, and source definitions.</p><p>Signal categories absent from the source are excluded, and the remaining weights are rescaled. Formula-text matches do not prove equal calculation results. Per-signal coverage contributions are not saved, so cataloged differences do not directly explain a weighted score gap.</p></details></section><section class="score-section"><h4>Overall similarity: '+score(pair.composite)+'</h4><p class="plain">'+SCORE_DIFFERENCE+'</p><dl class="evidence-grid">'+items.map(function(item){return metricHTML(item[0],item[1]);}).join('')+'</dl><p class="plain">Name, link, and source overlap means shared unique entries divided by all unique entries across both models. Text similarity measures resemblance, not the percentage of matching formulas.</p><details class="method"><summary>Similarity calculation</summary><p>The overall score is a weighted blend of five overlap signals and one TF-IDF text-similarity signal. DAX text similarity compares measure names and normalized formula text. For models without measures, structural or model names are used instead.</p><p>An overlap category empty in both models contributes zero, not a perfect match. Matching names, relationship endpoints, and source-definition text do not establish matching data types, relationship settings, or data values. These are metadata scores, not confidence ratings.</p></details></section>';
  }
  function candidateHTML(item,index){
    var pair=item.pair,category=item.category,context=coverageContext(pair),meta=[];
    if(pair.crossWorkspace){meta.push('Cross-workspace comparison');}if(pair.sameName){meta.push('Same model name');}
    var leftId=pair.idA,rightId=pair.idB,primary='<span>Overall similarity</span><strong>'+score(pair.composite)+'</strong>';
    var meaning='A weighted comparison of cataloged names, structure, source definitions, and formula text; not proof of matching data or calculation results.';
    var secondary='';
    if(category.kind==='containment'){
      if(context.strongest){leftId=context.strongest.sourceId;rightId= context.strongest.targetId;}
      var directions=context.both?context.directions:(context.strongest?[context.strongest]:[]);
      primary='<span>Coverage score'+(context.both?'s':'')+'</span>'+(directions.length?directions.map(function(direction){return '<div class="direction-score"><strong>'+score(direction.value)+'</strong><span>'+esc(directionLabel(direction))+'</span></div>';}).join(''):'<strong>Unavailable</strong><span>Direction not available</span>');
      meaning=COVERAGE_MEANING;
      secondary='<div class="secondary-score"><strong>Overall similarity: '+score(pair.composite)+'</strong><div class="plain">'+SCORE_DIFFERENCE+'</div></div>';
    }
    return '<article class="card candidate"><div class="relationship">'+category.label+'</div><h3 class="finding">'+esc(relationshipText(pair,category))+'</h3><div class="candidate-main"><div><div class="identity-row">'+identity(leftId)+'<div class="relation-word">compared with</div>'+identity(rightId)+'</div><div class="plain">'+meaning+'</div>'+secondary+(meta.length?'<div class="metadata">'+meta.join(' · ')+'</div>':'')+'</div><div class="score">'+primary+'</div></div><p class="caution">'+REVIEW_CAUTION+'</p><div class="candidate-actions"><button class="button disclosure" data-evidence="evidence-'+index+'" aria-expanded="false" aria-controls="evidence-'+index+'">Score breakdown</button><button class="button primary" data-compare-a="'+esc(leftId)+'" data-compare-b="'+esc(rightId)+'" data-open-section="'+relevantSection(pair)+'">Compare model details</button></div><div class="evidence" id="evidence-'+index+'" hidden>'+evidenceHTML(pair)+'</div></article>';
  }
  function reviewQueueHTML(){
    var all=candidates(),filtered=all.filter(function(item){if(state.filter!=='all'&&item.category.kind!==state.filter){return false;}var query=state.search.trim().toLowerCase();if(!query){return true;}var pair=item.pair;return [model(pair.idA).name,model(pair.idA).workspace,model(pair.idB).name,model(pair.idB).workspace].join(' ').toLowerCase().indexOf(query)>=0;});
    return filtered.length?filtered.map(candidateHTML).join(''):'<div class="empty">No scored pairs match the current filters and review thresholds. This does not prove that no similar models exist.</div>';
  }
  function reviewHTML(){
    var all=candidates(),groups=buildGroups();
    var counts={duplicate:0,containment:0,overlap:0};all.forEach(function(item){counts[item.category.kind]++;});
    var filters=[['all','All',all.length],['duplicate',LABELS.duplicate,counts.duplicate],['containment',LABELS.containment,counts.containment],['overlap',LABELS.overlap,counts.overlap]];
    return '<h2>Review candidates</h2><p class="intro">Possible duplicates have high overall similarity. Model coverage asks how much of one model is represented in another. Shared structure marks other pairs meeting the similarity cutoff. Each pair is counted once, in that order.</p><div class="stats"><div class="stat"><strong>'+DATA.summary.models+'</strong><span>models in the catalog</span></div><div class="stat"><strong>'+all.length+'</strong><span>model pairs meeting review cutoffs</span></div><div class="stat"><strong>'+groups.length+'</strong><span>groups linked by possible duplicates</span></div></div>'+reportStatusHTML()+'<div class="toolbar"><input class="search" type="text" aria-label="Search review candidates" placeholder="Search models or workspaces" value="'+esc(state.search)+'"><div class="filter-row" aria-label="Candidate filters">'+filters.map(function(filter){return '<button class="filter'+(state.filter===filter[0]?' active':'')+'" data-filter="'+filter[0]+'" aria-pressed="'+(state.filter===filter[0])+'">'+filter[1]+' '+filter[2]+'</button>';}).join('')+'</div></div><div class="queue">'+reviewQueueHTML()+'</div>';
  }

  function groupOptions(members,selected){return members.map(function(id){var entry=model(id);return '<option value="'+esc(id)+'"'+(id===selected?' selected':'')+'>'+esc(entry.name)+' (Workspace: '+esc(entry.workspace)+')</option>';}).join('');}
  function groupsHTML(){
    var groups=buildGroups();
    var intro='<h2>Groups of possible duplicates</h2><p class="intro">Models linked by highly similar pairs. Not every pair in a group is equally similar: a model can join through another member. A group is a review aid, not an approved consolidation.</p>';
    if(!groups.length){return intro+'<div class="empty">No groups meet the current possible-duplicate cutoff.</div>';}
    return intro+reportStatusHTML()+'<div class="groups">'+groups.map(function(group,index){
      var key='g'+index,selected=state.groupSelections[key]||{a:group.strongest.idA,b:group.strongest.idB};state.groupSelections[key]=selected;
      return '<article class="card group"><div class="group-head"><h3>Group '+(index+1)+'</h3><span class="muted">'+group.members.length+' models linked by possible duplicates</span></div><p class="plain">Highest overall similarity in this group: '+esc(pairModelLabel(group.strongest.idA,group.strongest.idB))+' and '+esc(pairModelLabel(group.strongest.idB,group.strongest.idA))+' ('+score(group.strongest.composite)+').</p><ul class="group-members">'+group.members.map(function(id){var entry=model(id);return '<li><span class="model-name">'+esc(entry.name)+'<span class="report-count"> · '+esc(reportCountText(id))+'</span></span><span class="workspace">Workspace: '+esc(entry.workspace)+'</span></li>';}).join('')+'</ul><div class="group-pickers"><label>First model<select data-group="'+key+'" data-group-side="a">'+groupOptions(group.members,selected.a)+'</select></label><label>Second model<select data-group="'+key+'" data-group-side="b">'+groupOptions(group.members,selected.b)+'</select></label><button class="button primary" data-group-compare="'+key+'">Compare selected models</button></div></article>';
    }).join('')+'</div>';
  }

  function indexBy(list){var out={};(list||[]).forEach(function(x){out[x.key]=x;});return out;}
  function unionKeys(a,b){var out={};Object.keys(a).forEach(function(k){out[k]=1;});Object.keys(b).forEach(function(k){out[k]=1;});return Object.keys(out).sort();}
  function statusOf(key,a,b){return a[key]&&b[key]?'shared':(a[key]?'onlyA':'onlyB');}
  function statusLabel(status,firstName,secondName){return status==='onlyA'?'Only in '+firstName:(status==='onlyB'?'Only in '+secondName:(status==='changed'?'Different formula text':'Matching entry'));}
  function sectionHTML(id,title,summary,rows){var open=!!state.openSections[id];return '<section class="diff-section"><button class="section-button" data-section="'+id+'" aria-expanded="'+open+'" aria-controls="section-'+id+'"><span>'+title+'</span><span class="muted">'+summary+'</span></button><div class="section-body" id="section-'+id+'"'+(open?'':' hidden')+'>'+((rows&&rows.length)?rows.join(''):'<div class="muted">No cataloged entries to show.</div>')+'</div></section>';}
  function rowHTML(st,text,aName,bName,detail){return '<div class="diff-row '+st+'"><span class="status">'+esc(statusLabel(st,aName,bName))+'</span><span>'+text+'</span></div>'+(detail||'');}
  function compareData(a,b,firstName,secondName){
    firstName=firstName||a.name;secondName=secondName||b.name;
    var result={sections:{},shared:0,differences:0};
    var ta=indexBy(a.tables),tb=indexBy(b.tables),ca=indexBy(a.columns),cb=indexBy(b.columns),ma=indexBy(a.measures),mb=indexBy(b.measures),ra=indexBy(a.relationships),rb=indexBy(b.relationships),da=indexBy(a.datasources),db=indexBy(b.datasources);
    function simple(id,left,right,format){var rows=[],shared=0,diff=0;unionKeys(left,right).forEach(function(k){var st=statusOf(k,left,right);if(st==='shared'){shared++;}else{diff++;}if(!state.cmpDiffOnly||st!=='shared'){rows.push(rowHTML(st,format(left[k]||right[k]),firstName,secondName));}});result.sections[id]={rows:rows,shared:shared,diff:diff};result.shared+=shared;result.differences+=diff;}
    simple('tables',ta,tb,function(x){return esc(x.name);});
    simple('columns',ca,cb,function(x){return esc(x.table)+'['+esc(x.name)+']';});
    var measureRows=[],measureShared=0,measureDiff=0;unionKeys(ma,mb).forEach(function(k){var ia=ma[k],ib=mb[k],st;if(ia&&ib){st=ia.daxHash===ib.daxHash?'shared':'changed';}else{st=ia?'onlyA':'onlyB';}if(st==='shared'){measureShared++;}else{measureDiff++;}if(!state.cmpDiffOnly||st!=='shared'){var item=ia||ib,detail='';if(st==='changed'){detail='<div class="dax"><strong>'+esc(firstName)+'</strong>\n'+esc(daxText(ia))+'\n\n<strong>'+esc(secondName)+'</strong>\n'+esc(daxText(ib))+'</div>';}else if(st!=='shared'){detail='<div class="dax">'+esc(daxText(item))+'</div>';}measureRows.push(rowHTML(st,esc(item.name),firstName,secondName,detail));}});result.sections.measures={rows:measureRows,shared:measureShared,diff:measureDiff};result.shared+=measureShared;result.differences+=measureDiff;
    simple('relationships',ra,rb,function(x){return esc(x.from)+' → '+esc(x.to);});
    simple('datasources',da,db,function(x){return esc(x.name);});
    return result;
  }
  function relationshipSummary(pair){if(!pair){return 'Not scored';}var category=classify(pair);return category?category.label:'Below review cutoffs or unavailable';}
  function containmentSummary(pair){
    if(!pair){return 'Not scored';}
    var context=coverageContext(pair);
    if(!context.strongest){return 'Directional coverage unavailable';}
    if(context.directions.some(function(direction){return !validScore(direction.value);})){return 'Coverage comparison is incomplete: one directional score is unavailable.';}
    if(context.both){return 'Both models meet the coverage threshold ('+score(state.thresholds.containment)+').';}
    if(context.qualifies){return 'One direction meets the coverage threshold ('+score(state.thresholds.containment)+').';}
    return 'Neither direction meets the coverage threshold ('+score(state.thresholds.containment)+').';
  }
  function selectOptions(selected){return MLIST.map(function(entry){return '<option value="'+esc(entry.id)+'"'+(entry.id===selected?' selected':'')+'>'+esc(entry.name)+' (Workspace: '+esc(entry.workspace)+')</option>';}).join('');}
  function compareHTML(){
    if(MLIST.length<2){return '<h2>Compare models</h2><div class="empty">At least two cataloged models are required for a comparison.</div>';}
    var first=model(state.cmpA),second=model(state.cmpB),pair=pairMap()[pairKey(state.cmpA,state.cmpB)];
    var controls='<div class="compare-controls"><label>First model<select data-compare-select="a">'+selectOptions(state.cmpA)+'</select></label><button class="button" data-swap aria-label="Swap selected models">Swap models</button><label>Second model<select data-compare-select="b">'+selectOptions(state.cmpB)+'</select></label>';
    if(state.cmpA===state.cmpB){return '<h2>Compare models</h2>'+controls+'</div><div class="empty">Choose two different models.</div>';}
    var diff=compareData(first,second,pairModelLabel(state.cmpA,state.cmpB),pairModelLabel(state.cmpB,state.cmpA)),category=pair?classify(pair):null;
    function sec(id,title,rule){var entry=diff.sections[id]||{rows:[],shared:0,diff:0};var rows=['<p class="plain">'+rule+'</p>'].concat(entry.rows.length?entry.rows:['<div class="muted">'+(state.cmpDiffOnly?'No differing entries in this category.':'No cataloged entries in this category.')+'</div>']);return sectionHTML(id,title,entry.shared+' matching · '+entry.diff+' different',rows);}
    return '<h2>Compare models</h2><p class="intro">Matches refer to cataloged names or definitions, not proof of matching data or behavior.</p>'+controls+'<label><input type="checkbox" data-diff-only'+(state.cmpDiffOnly?' checked':'')+'> Show only differences</label></div><div class="compare-summary"><div class="relationship">'+esc(relationshipSummary(pair))+'</div><h3 class="finding">'+esc(relationshipText(pair,category))+'</h3><div class="identity-row">'+identity(state.cmpA)+'<div class="relation-word">compared with</div>'+identity(state.cmpB)+'</div><div class="summary-grid"><div class="summary-item"><span>Overall similarity</span><strong>'+(pair?score(pair.composite):'Not scored')+'</strong><p class="plain">'+SCORE_DIFFERENCE+'</p></div><div class="summary-item"><span>Model coverage</span>'+coverageRowsHTML(pair)+'<div class="plain">'+esc(containmentSummary(pair))+'</div></div><div class="summary-item"><span>Cataloged entries</span><strong>'+diff.shared+' matching · '+diff.differences+' different</strong><p class="plain">These counts are not percentages and do not directly explain a weighted score gap.</p></div></div><p class="plain">'+COVERAGE_MEANING+'</p><p class="caution">Scores do not verify data values, calculation results, security, refresh behavior, or replacement safety.</p></div>'+sectionHTML('scores','Score breakdown','Coverage and overall similarity',[evidenceHTML(pair)])+reportCompareHTML()+sec('tables','Tables','Table names are matched after normalization; data and table properties are not compared.')+sec('columns','Columns','Column names and their table names are matched; matching names do not establish matching data types or values.')+sec('measures','Measures and formula text','Matching entries have the same normalized measure name and formula text. Different formula text does not necessarily mean different results, and matching text does not prove equal results.')+sec('relationships','Relationship links','Matches compare the source and target table/column endpoints, not all relationship properties.')+sec('datasources','Source definitions','Matches compare normalized source-definition text, not the contents of the connected data sources.');
  }

  function mapHTML(){
    if(!MLIST.length){return '<h2>Overall similarity map</h2><div class="empty">No cataloged models are available.</div>';}
    var pmap=pairMap(),head='<tr><th></th>'+MLIST.map(function(entry){var label=entry.name+' (Workspace: '+entry.workspace+')';return '<th class="col" scope="col" title="'+esc(label)+'">'+esc(label)+'</th>';}).join('')+'</tr>';
    var rows=MLIST.map(function(rowModel,rowIndex){var cells=MLIST.map(function(columnModel,columnIndex){
      var rowLabel=rowModel.name+' (Workspace: '+rowModel.workspace+')',columnLabel=columnModel.name+' (Workspace: '+columnModel.workspace+')',pairLabel=rowLabel+' and '+columnLabel;
      if(rowIndex===columnIndex){return '<td><span class="matrix-cell diagonal" title="'+esc(rowLabel)+': same model" aria-label="'+esc(rowLabel)+': same model">—</span></td>';}
      var pair=pmap[pairKey(rowModel.id,columnModel.id)];
      if(!pair){return '<td><span class="matrix-cell unscored" title="'+esc(pairLabel)+': Not scored" aria-label="'+esc(pairLabel)+': Not scored"></span></td>';}
      var tier=tierAt(pair.composite),available=validScore(pair.composite),style=!available?'unavailable':(tier==='duplicate'?'duplicate':(tier==='similar'?'high':'low'));
      return '<td><button class="matrix-cell '+style+'" data-map-a="'+esc(rowModel.id)+'" data-map-b="'+esc(columnModel.id)+'" title="'+esc(pairLabel)+': Overall similarity '+score(pair.composite)+'" aria-label="Compare '+esc(pairLabel)+', overall similarity '+score(pair.composite)+'">'+(available?Math.round(pair.composite*100)+'%':'?')+'</button></td>';
    }).join('');return '<tr><th scope="row" title="'+esc(rowModel.name)+' (Workspace: '+esc(rowModel.workspace)+')">'+esc(rowModel.name)+'<div class="workspace">Workspace: '+esc(rowModel.workspace)+'</div></th>'+cells+'</tr>';}).join('');
    return '<h2>Overall similarity map</h2><p class="intro">Overall similarity between each pair, shown as percentages. Low similarity does not rule out high model coverage.</p><div class="map-note"><strong>Not scored is not zero:</strong> blank cells have no saved pair score, for example because name-based candidate filtering excluded the pair. A scored 0% is a real result; ? means the saved pair has no usable overall-similarity value.</div><div class="map-wrap"><table class="matrix" aria-label="Semantic model overall similarity map">'+head+rows+'</table><div class="legend" aria-label="Similarity map legend"><span><span class="legend-key duplicate"></span> '+LABELS.duplicate+'</span><span><span class="legend-key high"></span> '+LABELS.overlap+'</span><span><span class="legend-key low"></span> Below similarity cutoff (including 0%)</span><span><span class="legend-key unscored"></span> Not scored (blank)</span><span><span class="legend-key unavailable"></span> Unavailable (?)</span></div></div>';
  }

  function viewHTML(){var content=state.tab==='review'?reviewHTML():(state.tab==='groups'?groupsHTML():(state.tab==='compare'?compareHTML():(state.tab==='reports'?reportsHTML():mapHTML())));return '<div class="view" role="tabpanel" id="panel-'+state.tab+'" aria-labelledby="tab-'+state.tab+'">'+content+'</div>';}
  function footHTML(){return '<div class="foot">Current review cutoffs: '+LABELS.duplicate+' at '+score(state.thresholds.duplicate)+' overall similarity; '+LABELS.overlap+' at '+score(state.thresholds.similar)+' overall similarity; '+LABELS.containment+' at '+score(state.thresholds.containment)+' coverage in at least one direction. Existing scores only; unscored pairs remain unscored.</div>';}
  function render(){root.innerHTML=headerHTML()+settingsHTML()+tabsHTML()+viewHTML()+footHTML();applyTheme(false);wire();}
  function setTab(tab){state.tab=tab;render();}
  function openCompare(a,b,section){state.cmpA=String(a);state.cmpB=String(b);state.cmpDiffOnly=true;state.openSections={reports:true};if(section){state.openSections[section]=true;}setTab('compare');}
  function applyTheme(save){document.documentElement.setAttribute('data-theme',state.theme);if(save){try{localStorage.setItem('sms-theme',state.theme);}catch(e){}}}
  function applyThresholds(){state.thresholds=Object.assign({},state.draft);try{localStorage.setItem('sms-thresholds',JSON.stringify(state.thresholds));}catch(e){}render();}

  function wireCandidateActions(scope){
    scope.querySelectorAll('[data-evidence]').forEach(function(btn){btn.addEventListener('click',function(){var target=scope.querySelector('#'+btn.dataset.evidence),expanded=btn.getAttribute('aria-expanded')==='true';btn.setAttribute('aria-expanded',String(!expanded));if(target){target.hidden=expanded;}});});
    scope.querySelectorAll('[data-compare-a]').forEach(function(btn){btn.addEventListener('click',function(){openCompare(btn.dataset.compareA,btn.dataset.compareB,btn.dataset.openSection);});});
  }

  function wire(){
    var tabs=Array.from(root.querySelectorAll('[data-tab]'));
    tabs.forEach(function(button,index){
      button.addEventListener('click',function(){setTab(button.dataset.tab);});
      button.addEventListener('keydown',function(event){
        var next=index;
        if(event.key==='ArrowRight'){next=(index+1)%tabs.length;}
        else if(event.key==='ArrowLeft'){next=(index+tabs.length-1)%tabs.length;}
        else if(event.key==='Home'){next=0;}
        else if(event.key==='End'){next=tabs.length-1;}
        else{return;}
        event.preventDefault();setTab(tabs[next].dataset.tab);root.querySelector('[data-tab="'+state.tab+'"]').focus();
      });
    });
    var settings=root.querySelector('[data-settings]');if(settings){settings.addEventListener('click',function(){state.settingsOpen=!state.settingsOpen;render();root.querySelector('[data-settings]').focus();});}
    root.querySelectorAll('[data-theme-set]').forEach(function(btn){btn.addEventListener('click',function(){state.theme=btn.dataset.themeSet;applyTheme(true);render();});});
    root.querySelectorAll('[data-draft-range],[data-draft-number]').forEach(function(input){input.addEventListener('input',function(){var key=input.dataset.draftRange||input.dataset.draftNumber,value=Math.max(0,Math.min(1,parseFloat(input.value)/100));if(!Number.isFinite(value)){return;}state.draft[key]=value;var other=root.querySelector(input.dataset.draftRange?'[data-draft-number="'+key+'"]':'[data-draft-range="'+key+'"]');if(other){other.value=String(Math.round(value*10000)/100);}});});
    var apply=root.querySelector('[data-apply]');if(apply){apply.addEventListener('click',applyThresholds);}
    var reset=root.querySelector('[data-reset]');if(reset){reset.addEventListener('click',function(){state.draft=Object.assign({},DEFAULTS);applyThresholds();});}
    var search=root.querySelector('.search');if(search){search.addEventListener('input',function(){state.search=search.value;var queue=root.querySelector('.queue');if(queue){queue.innerHTML=reviewQueueHTML();wireCandidateActions(queue);}});}
    var reportSearch=root.querySelector('.report-search');if(reportSearch){reportSearch.addEventListener('input',function(){state.reportSearch=reportSearch.value;root.querySelector('.report-models').innerHTML=reportRowsHTML();});}
    var reportsOnly=root.querySelector('[data-reports-only]');if(reportsOnly){reportsOnly.addEventListener('change',function(){state.reportsOnly=reportsOnly.checked;root.querySelector('.report-models').innerHTML=reportRowsHTML();});}
    root.querySelectorAll('[data-filter]').forEach(function(btn){btn.addEventListener('click',function(){state.filter=btn.dataset.filter;render();});});
    wireCandidateActions(root);
    root.querySelectorAll('[data-group-side]').forEach(function(sel){sel.addEventListener('change',function(){var key=sel.dataset.group,side=sel.dataset.groupSide;state.groupSelections[key]=state.groupSelections[key]||{};state.groupSelections[key][side]=sel.value;});});
    root.querySelectorAll('[data-group-compare]').forEach(function(btn){btn.addEventListener('click',function(){var selected=state.groupSelections[btn.dataset.groupCompare];if(selected&&selected.a!==selected.b){openCompare(selected.a,selected.b,'tables');}});});
    root.querySelectorAll('[data-compare-select]').forEach(function(sel){sel.addEventListener('change',function(){if(sel.dataset.compareSelect==='a'){state.cmpA=sel.value;}else{state.cmpB=sel.value;}render();});});
    var swap=root.querySelector('[data-swap]');if(swap){swap.addEventListener('click',function(){var tmp=state.cmpA;state.cmpA=state.cmpB;state.cmpB=tmp;render();});}
    var only=root.querySelector('[data-diff-only]');if(only){only.addEventListener('change',function(){state.cmpDiffOnly=only.checked;render();});}
    root.querySelectorAll('[data-section]').forEach(function(button){button.addEventListener('click',function(){var id=button.dataset.section;state.openSections[id]=!state.openSections[id];render();root.querySelector('[data-section="'+id+'"]').focus();});});
    root.querySelectorAll('[data-map-a]').forEach(function(btn){btn.addEventListener('click',function(){openCompare(btn.dataset.mapA,btn.dataset.mapB,'tables');});});
  }

  render();
})();
</script>
"""

    app_json = json.dumps(app_data).replace("<", "\\u003c")
    displayHTML(app_template.replace("__APP_DATA__", app_json))

In [ ]:
render_results()